# Multi-Regional AeroMAPS Scenarios using GEMSEO MDAChain

This notebook demonstrates how to run multiple regional AeroMAPS scenarios within a single GEMSEO MDAChain using namespaces to isolate regional data.

## Approach

1. Create regional AeroMAPS processes to get properly configured disciplines
2. Apply GEMSEO namespaces to isolate regional I/O variables
3. Create an aggregation discipline for global metrics
4. Combine everything into a single MDAChain

## 1. Prepare Regional Data

In [ ]:
from aeromaps.utils.functions import create_partitioning

# Generate partitioning data for both regions
create_partitioning(file="data/region_france/aeroscope_data.csv", path="data/region_france")
create_partitioning(file="data/region_germany/aeroscope_data.csv", path="data/region_germany")
print("✓ Partitioning data generated for both regions")

## 2. Create Regional Processes and Extract Disciplines

In [ ]:
%matplotlib widget
from aeromaps import create_process
from copy import deepcopy

# Create processes to get properly configured disciplines
process_fr = create_process(configuration_file="data/region_france/config.yaml")
process_de = create_process(configuration_file="data/region_germany/config.yaml")

print(f"France:  region created, with {len(process_fr.disciplines)} disciplines")
print(f"Germany:  region created, with {len(process_de.disciplines)} disciplines")

## 3. Apply Namespaces to Disciplines

We use GEMSEO's namespace feature to prefix all I/O variables with a region identifier.

In [ ]:
def apply_namespace_to_discipline(discipline, namespace: str):
    """
    Apply namespace to all inputs and outputs of a discipline.

    Note: deepcopy is necessary because add_namespace_to_input/output()
    modifies the discipline in place. Without it, we'd modify the original
    disciplines stored in the process objects.
    """
    ns_disc = deepcopy(discipline)

    # Collect all variable names first for consistency
    input_names = list(ns_disc.input_grammar.names)
    output_names = list(ns_disc.output_grammar.names)

    # Apply namespace to all inputs and outputs together
    for name in input_names:
        ns_disc.add_namespace_to_input(name, namespace)
    for name in output_names:
        ns_disc.add_namespace_to_output(name, namespace)

    # Update discipline name to include region
    ns_disc.name = f"{namespace}_{ns_disc.name}"

    return ns_disc


# Create namespaced discipline lists
fr_disciplines = [apply_namespace_to_discipline(d, "FR") for d in process_fr.disciplines]
de_disciplines = [apply_namespace_to_discipline(d, "DE") for d in process_de.disciplines]

print(f"\nNamespaced France disciplines: {len(fr_disciplines)}")
print(f"Namespaced Germany disciplines: {len(de_disciplines)}")

# Verify namespace application
print(f"\nExample FR discipline: {fr_disciplines[0].name}")
print(f"  Outputs: {list(fr_disciplines[0].output_grammar.names)[:3]}")
print(f"\nExample DE discipline: {de_disciplines[0].name}")
print(f"  Outputs: {list(de_disciplines[0].output_grammar.names)[:3]}")

## 4. Create Aggregation Discipline

Create an `AeroMAPSModel`-based discipline that aggregates regional outputs into global metrics.

In [ ]:
import pandas as pd
import numpy as np
from aeromaps.models.base import AeroMAPSModel
from aeromaps.core.gemseo import AeroMAPSCustomModelWrapper


class RegionalAggregator(AeroMAPSModel):
    """
    Aggregates CO2 emissions from multiple regional processes into a global metric.

    TODO: Later, use a config file to specify which metrics to aggregate
    and which operation to use (sum, weighted average, etc.)
    """

    def __init__(
        self,
        name: str,
        regions: list,
        parameters=None,
    ):
        super().__init__(name=name, parameters=parameters, model_type="custom")

        self.regions = regions

        # Build input names: {region}:co2_emissions for each region
        self.input_names = {}
        for region in regions:
            self.input_names[f"{region}:co2_emissions"] = pd.Series(dtype=float)

        # Output: global:co2_emissions
        self.output_names = {"global:co2_emissions": pd.Series(dtype=float)}

    def compute(self, input_data: dict) -> dict:
        """Aggregate regional CO2 emissions by linear sum."""
        total = None

        for region in self.regions:
            key = f"{region}:co2_emissions"
            if key in input_data:
                value = input_data[key]
                if total is None:
                    total = value.copy() if hasattr(value, "copy") else value
                else:
                    total = total + value

        return {"global:co2_emissions": total}


REGIONS = ["FR", "DE"]

# Create aggregator model and wrap as discipline
aggregator_model = RegionalAggregator(
    name="RegionalAggregator",
    regions=REGIONS,
    parameters=process_fr.parameters,
)

aggregator_discipline = AeroMAPSCustomModelWrapper(model=aggregator_model)

print(f"Aggregator inputs: {list(aggregator_discipline.input_grammar.names)}")
print(f"Aggregator outputs: {list(aggregator_discipline.output_grammar.names)}")

## 5. Build Combined MDAChain

Combine all regional disciplines and the aggregator into a single GEMSEO MDAChain.

In [ ]:
from gemseo.mda.mda_chain import MDAChain

# Combine all disciplines
all_disciplines = fr_disciplines + de_disciplines + [aggregator_discipline]

print(f"Total disciplines: {len(all_disciplines)}")
print(f"  - France: {len(fr_disciplines)}")
print(f"  - Germany: {len(de_disciplines)}")
print(f"  - Aggregator: 1")

In [ ]:
# Build the combined MDAChain
multi_regional_mda = MDAChain(
    disciplines=all_disciplines,
    tolerance=1e-5,
    initialize_defaults=True,
    inner_mda_name="MDAGaussSeidel",
    log_convergence=True,
)

print(f"\n✓ Multi-regional MDAChain created")
print(f"  Input variables: {len(multi_regional_mda.input_grammar.names)}")
print(f"  Output variables: {len(multi_regional_mda.output_grammar.names)}")

## 6. Prepare Input Data and Execute

In [ ]:
# Build namespaced input data from both processes' parameters
def build_namespaced_inputs(process, namespace: str) -> dict:
    """Build namespaced input dictionary from process parameters."""
    input_data = {}
    params_dict = process.parameters.to_dict()

    for key, value in params_dict.items():
        namespaced_key = f"{namespace}:{key}"
        input_data[namespaced_key] = value

    return input_data


# Combine inputs from both regions
input_data = {}
input_data.update(build_namespaced_inputs(process_fr, "FR"))
input_data.update(build_namespaced_inputs(process_de, "DE"))

print(f"Total input variables: {len(input_data)}")
print(f"Sample FR input: FR:rpk_init -> {type(input_data.get('FR:rpk_init', 'N/A'))}")
print(f"Sample DE input: DE:rpk_init -> {type(input_data.get('DE:rpk_init', 'N/A'))}")

In [ ]:
# Execute the combined MDAChain
print("Executing multi-regional MDAChain...")
result = multi_regional_mda.execute(input_data=input_data)
print("✓ Execution complete")

## 7. Examine Results

In [ ]:
# Access regional and global outputs
local_data = multi_regional_mda.local_data

years = [2020, 2030, 2040, 2050]

print("=" * 60)
print("MULTI-REGIONAL RESULTS FROM COMBINED MDAChain")
print("=" * 60)

# CO2 Emissions comparison
print("\nCO2 EMISSIONS (Mt):")
print("-" * 40)

data = {"Year": years}

if "FR:co2_emissions" in local_data:
    data["France"] = [local_data["FR:co2_emissions"][y] / 1e9 for y in years]
if "DE:co2_emissions" in local_data:
    data["Germany"] = [local_data["DE:co2_emissions"][y] / 1e9 for y in years]
if "global:co2_emissions" in local_data:
    data["GLOBAL"] = [local_data["global:co2_emissions"][y] / 1e9 for y in years]

df = pd.DataFrame(data).set_index("Year")
display(df)

In [ ]:
# Verify aggregation correctness for CO2
print("\nVERIFICATION - CO2 emissions 2050:")
print("-" * 40)

fr_val = local_data["FR:co2_emissions"][2050] if "FR:co2_emissions" in local_data else 0
de_val = local_data["DE:co2_emissions"][2050] if "DE:co2_emissions" in local_data else 0
global_val = local_data["global:co2_emissions"][2050] if "global:co2_emissions" in local_data else 0

print(f"  FR + DE = {(fr_val + de_val) / 1e9:.2f} Mt")
print(f"  Global  = {global_val / 1e9:.2f} Mt")
print(f"  Match: {np.isclose(fr_val + de_val, global_val)}")

## 8. Summary

We successfully:
1. Created two regional AeroMAPS processes with proper discipline configurations
2. Applied GEMSEO namespaces to isolate regional I/O (`FR:`, `DE:`)
3. Built an `AeroMAPSModel`-based aggregation discipline
4. Combined everything into a single `MDAChain` that executes all regions together
5. Verified that global aggregates match the sum of regional values

In [ ]:
# Cleanup
from aeromaps.utils.functions import clean_notebooks_on_tests

clean_notebooks_on_tests(globals())